## Generates the NIL Valuation for teams in the Big 12

In [12]:
import pandas as pd
df = pd.read_csv('the-nil-standard-basketball-team-roster-spend-2026-11-02.csv')
big12_df = df[df["conference"] == "Big 12"].copy()

big12_teams = big12_df["team"].tolist()

big12_df = big12_df.sort_values(
    by="est_roster_spend_usd",
    ascending=False
)

print(big12_df[["team", "est_roster_spend_usd"]])

              team  est_roster_spend_usd
10          Kansas              19250000
14      Texas Tech              17500000
16         Arizona              17000000
19         Houston              16500000
22      Cincinnati              16000000
25  Oklahoma State              15750000
43   West Virginia              12750000
47          Baylor              11500000
49      Iowa State              11000000
53             BYU              10500000
55   Arizona State               9750000
63             TCU               8750000
65            Utah               8500000
69             UCF               8250000
78        Colorado               6250000
79    Kansas State               6000000


## generate links for getting teams 

In [13]:

big12_df["sports_reference_url"] = (
    "https://www.sports-reference.com/cbb/schools/"
    + big12_df["team"].str.lower().str.replace(" ", "-", regex=False)
    + "/men/2027.html"
)

In [14]:
big12_df['sports_reference_url'].to_list()

['https://www.sports-reference.com/cbb/schools/kansas/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/texas-tech/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/arizona/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/houston/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/cincinnati/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/oklahoma-state/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/west-virginia/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/baylor/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/iowa-state/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/byu/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/arizona-state/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/tcu/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/utah/men/2027.html',
 'https://www.sports-reference.com/cbb/schools/ucf/men/2027.ht

## Downloading the roster data for each team in big 12

In [18]:
import pandas as pd
import requests
import time
from io import StringIO
from pathlib import Path

# Special Sports Reference team names
team_name_mapping = {
    "BYU": "brigham-young",
    "TCU": "texas-christian",
    "UCF": "central-florida"
}

# Create folder for roster CSVs
output_dir = Path("big12_rosters")
output_dir.mkdir(exist_ok=True)

for _, row in big12_df.iterrows():
    team = row["team"]

    # Get correct Sports Reference team slug
    team_slug = team_name_mapping.get(
        team,
        team.lower().replace(" ", "-")
    )

    # Construct URL for 2026-27 season
    url = (
        f"https://www.sports-reference.com/cbb/"
        f"schools/{team_slug}/men/2027.html"
    )

    try:
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=30
        )
        response.raise_for_status()

        # Extract first HTML table
        tables = pd.read_html(StringIO(response.text))
        df_roster = tables[0]

        # Save with lowercase and underscores
        filename = output_dir / f"{team.lower().replace(' ', '_')}.csv"
        df_roster.to_csv(filename, index=False)

        print(f"Saved {filename}")

    except Exception as e:
        print(f"Error downloading {team}: {e}")

    # Delay requests to avoid overwhelming the website
    time.sleep(5)

print("Finished downloading Big 12 team data!")

Saved big12_rosters/kansas.csv
Saved big12_rosters/texas_tech.csv
Saved big12_rosters/arizona.csv
Saved big12_rosters/houston.csv
Saved big12_rosters/cincinnati.csv
Saved big12_rosters/oklahoma_state.csv
Saved big12_rosters/west_virginia.csv
Saved big12_rosters/baylor.csv
Saved big12_rosters/iowa_state.csv
Saved big12_rosters/byu.csv
Saved big12_rosters/arizona_state.csv
Saved big12_rosters/tcu.csv
Saved big12_rosters/utah.csv
Saved big12_rosters/ucf.csv
Saved big12_rosters/colorado.csv
Saved big12_rosters/kansas_state.csv
Finished downloading Big 12 team data!


## Download the totals for big 12 teams 

In [20]:
import pandas as pd
import time
from pathlib import Path

# ---------------------------------------
# 1. Load NIL data
# ---------------------------------------

df = pd.read_csv(
    "the-nil-standard-basketball-team-roster-spend-2026-11-02.csv"
)

big12_df = df[df["conference"] == "Big 12"].copy()

# Special Sports Reference team names
team_name_mapping = {
    "BYU": "brigham-young",
    "TCU": "texas-christian",
    "UCF": "central-florida"
}

# Create folder
TOTALS_DIR = Path("big12_totals")
TOTALS_DIR.mkdir(exist_ok=True)

# ---------------------------------------
# 2. Download all Big 12 totals
# ---------------------------------------

for _, row in big12_df.iterrows():

    team = row["team"]

    slug = team_name_mapping.get(
        team,
        team.lower().replace(" ", "-")
    )

    url = (
        f"https://www.sports-reference.com/cbb/"
        f"schools/{slug}/men/2027.html"
    )

    try:
        # Read all HTML tables
        tables = pd.read_html(url)

        matching_tables = []

        # Find tables containing player statistics
        for table in tables:

            # Flatten multi-level column names
            if isinstance(table.columns, pd.MultiIndex):
                table.columns = table.columns.get_level_values(-1)

            required_columns = {
                "Player", "G", "GS", "MP",
                "PTS", "TRB", "AST"
            }

            if required_columns.issubset(table.columns):

                if (
                    table["Player"]
                    .astype(str)
                    .str.strip()
                    .eq("Team Totals")
                    .any()
                ):
                    matching_tables.append(table.copy())

        # Select second matching table (Totals)
        if len(matching_tables) < 2:
            print(f"Totals table not found: {team}")
            continue

        totals = matching_tables[1].copy()

        # ---------------------------------------
        # 3. Clean rows (keep ALL columns)
        # ---------------------------------------

        totals = totals[
            totals["Player"].notna() &
            (totals["Player"] != "Team Totals") &
            (totals["Player"] != "Player")
        ].copy()

        # ---------------------------------------
        # 4. Save CSV
        # ---------------------------------------

        filename = (
            TOTALS_DIR /
            f"{team.lower().replace(' ', '_')}.csv"
        )

        totals.to_csv(filename, index=False)

        print(
            f"Saved: {filename} "
            f"({len(totals)} players, {len(totals.columns)} columns)"
        )

    except Exception as e:
        print(f"Error downloading {team}: {e}")

        if "403" in str(e) or "429" in str(e):
            print("Stopping due to access restrictions.")
            break

    time.sleep(5)

print("Finished downloading Big 12 totals!")

Saved: big12_totals/kansas.csv (15 players, 29 columns)
Saved: big12_totals/texas_tech.csv (12 players, 29 columns)
Saved: big12_totals/arizona.csv (11 players, 29 columns)
Saved: big12_totals/houston.csv (14 players, 29 columns)
Saved: big12_totals/cincinnati.csv (14 players, 29 columns)
Saved: big12_totals/oklahoma_state.csv (13 players, 29 columns)
Saved: big12_totals/west_virginia.csv (13 players, 29 columns)
Saved: big12_totals/baylor.csv (15 players, 29 columns)
Saved: big12_totals/iowa_state.csv (13 players, 29 columns)
Saved: big12_totals/byu.csv (12 players, 29 columns)
Saved: big12_totals/arizona_state.csv (12 players, 29 columns)
Saved: big12_totals/tcu.csv (14 players, 29 columns)
Saved: big12_totals/utah.csv (11 players, 29 columns)
Saved: big12_totals/ucf.csv (13 players, 29 columns)
Saved: big12_totals/colorado.csv (13 players, 29 columns)
Saved: big12_totals/kansas_state.csv (13 players, 29 columns)
Finished downloading Big 12 totals!
